In [61]:
pip install ucimlrepo

In [62]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

In [63]:
df = fetch_ucirepo(id=275)
X = df.data.features
y = df.data.targets

In [66]:
import numpy as np

class MyTree:
    def __init__(self, max_depth=5):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        self.n_features = X.shape[1]
        self.tree = self.build(X, y, 0)

    def build(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            return self.leaf_value(y)

        best_feat, best_thr = self.best_split(X, y)
        if best_feat is None:
            return self.leaf_value(y)

        left_mask = X[:, best_feat] <= best_thr
        right_mask = X[:, best_feat] > best_thr

        if sum(left_mask) == 0 or sum(right_mask) == 0:
            return self.leaf_value(y)

        return {
            'feat': best_feat,
            'thr': best_thr,
            'left': self.build(X[left_mask], y[left_mask], depth + 1),
            'right': self.build(X[right_mask], y[right_mask], depth + 1)
        }

    def leaf_value(self, y):
        if len(y.shape) > 1:
            return np.mean(y)
        else:
            return np.bincount(y.astype(int)).argmax()

    def best_split(self, X, y):
        best_gain = -1
        best_feat, best_thr = None, None

        for f in range(self.n_features):
            for thr in np.unique(X[:, f]):
                left_mask = X[:, f] <= thr
                right_mask = X[:, f] > thr

                if sum(left_mask) == 0 or sum(right_mask) == 0:
                    continue

                gain = self.gain(y, y[left_mask], y[right_mask])

                if gain > best_gain:
                    best_gain = gain
                    best_feat = f
                    best_thr = thr

        return best_feat, best_thr

    def gain(self, y, y_left, y_right):
        p_left = len(y_left) / len(y)
        p_right = 1 - p_left

        parent_impurity = self.impurity(y)
        left_impurity = self.impurity(y_left)
        right_impurity = self.impurity(y_right)

        return parent_impurity - p_left * left_impurity - p_right * right_impurity

    def impurity(self, y):
        if len(y.shape) > 1:
            return np.var(y)
        else:
            _, counts = np.unique(y, return_counts=True)
            probs = counts / len(y)
            return 1 - np.sum(probs ** 2)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict_row(x, self.tree) for x in X])

    def _predict_row(self, x, node):
        if not isinstance(node, dict):
            return node

        if x[node['feat']] <= node['thr']:
            return self._predict_row(x, node['left'])
        else:
            return self._predict_row(x, node['right'])

In [67]:
X_numeric = X.select_dtypes(include=[np.number])
X = X_numeric.fillna(X_numeric.mean())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MyTree(max_depth=10)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"R2: {r2_score(y_test, predictions):.4f}")
print(f"MSE: {mean_squared_error(y_test, predictions):.4f}")

R2: 0.6054
MSE: 12495.0429
